# 推理 Infra 学习实录 07：四种推理路径的统一横评

> 归档状态（GitHub 发布副本）：横评方案，尚无 48 条同机实测记录，不提供性能排名。 所有 ModelScope 发布 URL 尚未知，不表示已发布。
> [配套文章](article.md) · [上一篇](../06/article.md)。Notebook 输出与执行计数已清空。
> 版本边界：nano-vLLM 安装与源码说明对齐已核对 commit `bb823b3e06983d71485a8e1f23715ebd87d98ef8`；已有 checkout 不自动覆盖，版本不符时应另选空目录。
> 路径说明：`/mnt/workspace` 是 ModelScope 平台缓存示例，请按实际环境调整，模型目录以 `snapshot_download()` 返回值为准。

这份 Notebook 为 Transformers、nano-vLLM、mini-SGLang 和 vLLM 建立统一 benchmark 口径。离线生成与在线服务分开统计，未来结果必须来自同一 GPU 上的实际运行；当前只有方案，没有 48 条实测。

默认不启动框架 benchmark 子进程；设置环境变量 `RUN_FRAMEWORK_BENCHMARKS=1` 后才会顺序执行 48 组独立进程实验。依赖安装与 `snapshot_download()` 单元仍可能联网并写入本地缓存，不受该开关控制。如果只检查编排逻辑，请使用已装好 pandas 的 kernel、跳过安装和下载单元，并手动将 `model_dir` 设为已有本地模型目录。

## 1. 环境检查

GPU、driver、CUDA 和 PyTorch 版本属于 benchmark 结果的一部分。

In [ ]:
!nvidia-smi || true

In [ ]:
import platform
import sys

print("Platform:", platform.platform())
print("Python:", sys.version)
try:
    import torch
    print("PyTorch:", torch.__version__)
    print("Torch CUDA:", torch.version.cuda)
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
except ImportError:
    print("PyTorch is not installed in the orchestration kernel.")

## 2. Notebook 依赖与模型下载

Notebook kernel 只负责编排和汇总。四个推理框架分别安装在独立环境中。

In [ ]:
%pip install -q modelscope pandas

In [ ]:
from modelscope import snapshot_download

model_dir = snapshot_download(
    "Qwen/Qwen3-0.6B",
    cache_dir="/mnt/workspace/.cache/modelscope",
)
print("model_dir:", model_dir)

## 3. workload 矩阵

四组 case 分别覆盖 smoke、混合、decode-heavy 和 prefill-heavy。

In [ ]:
import pandas as pd

cases = pd.DataFrame([
    ("A", 4, 128, 128, "smoke"),
    ("B", 16, 512, 256, "mixed"),
    ("C", 64, 128, 512, "decode-heavy"),
    ("D", 16, 2048, 128, "prefill-heavy"),
], columns=["case", "num_requests", "input_len", "output_len", "purpose"])
cases

## 4. 独立环境

显式设置 `TRANSFORMERS_PYTHON`、`NANOVLLM_PYTHON`、`MINISGL_PYTHON`、`VLLM_PYTHON`，指向四个独立环境中的 Python 可执行文件；不回退到当前 kernel。不要在同一环境中反复升级或降级 PyTorch。

In [ ]:
from pathlib import Path
import os

repo_root = next(
    (parent for parent in (Path.cwd(), *Path.cwd().parents)
     if (parent / "nano-vllm/experiments/benchmarks").is_dir()),
    None,
)
if repo_root is None:
    raise FileNotFoundError("请从本合集目录内运行；未找到 nano-vllm/experiments/benchmarks。")

engine_env_vars = {
    "Transformers": "TRANSFORMERS_PYTHON",
    "nano-vLLM": "NANOVLLM_PYTHON",
    "mini-SGLang": "MINISGL_PYTHON",
    "vLLM": "VLLM_PYTHON",
}

def validate_engine_pythons():
    paths, issues = {}, []
    for engine, variable in engine_env_vars.items():
        value = os.environ.get(variable, "").strip()
        if not value:
            paths[engine] = None
            issues.append(f"缺少 {variable}")
            continue
        path = Path(value)
        paths[engine] = path
        if not path.is_absolute():
            issues.append(f"{variable} 必须是绝对路径：{value}")
        elif not path.is_file():
            issues.append(f"{variable} 不是文件：{value}")
        elif not os.access(path, os.X_OK):
            issues.append(f"{variable} 不可执行：{value}")
    configured = [path for path in paths.values() if path is not None and path.is_absolute()]
    # 保留 venv 的 Python 符号链接本身，不 resolve 到共享的基础解释器；
    # 但规范化父目录，防止目录别名或同环境 python/python3 冒充独立环境。
    executable_paths = [path.parent.resolve() / path.name for path in configured]
    environment_bins = [path.parent.resolve() for path in configured]
    if len(set(executable_paths)) != len(executable_paths) or len(set(environment_bins)) != len(environment_bins):
        issues.append("四个 Python 必须位于不同环境，路径和环境目录不得相同。")
    return paths, issues

engine_pythons, environment_issues = validate_engine_pythons()
print("repo_root:", repo_root)
print("Python paths:", engine_pythons)
for issue in environment_issues:
    print("待配置:", issue)
print("默认 dry-run；配置完成并显式设置 RUN_FRAMEWORK_BENCHMARKS=1 后才允许执行。")

依赖建议记录到各自环境的 `pip freeze`。mini-SGLang 使用固定 commit `9a91cfafe754aa85daee49998176275667eb58f2`。

nano-vLLM 在 `NANOVLLM_PYTHON` 指定的环境安装，不在编排 kernel 安装：

```bash
"${NANOVLLM_PYTHON:?请先配置独立环境}" -m pip install git+https://github.com/GeeeekExplorer/nano-vllm.git@bb823b3e06983d71485a8e1f23715ebd87d98ef8
```

源码固定不等于 torch/CUDA 依赖已锁定。

## 5. 生成统一命令

每个 case 独立运行三次。runner 将成功或失败记录追加到 JSONL。

每次重跑本节生成新的 UUID `run_id`，结果只写 `nano-vllm/experiments/benchmarks/results/<run_id>/offline_results.jsonl`；notes 精确记录 `session=<run_id> case=A run=1` 等组合，不向历史 JSONL 追加。

In [ ]:
import uuid
import shlex

runner_dir = repo_root / "nano-vllm/experiments/benchmarks"
runners = {
    "Transformers": runner_dir / "run_transformers_offline.py",
    "nano-vLLM": runner_dir / "run_nanovllm_offline.py",
    "mini-SGLang": runner_dir / "run_minisgl_offline.py",
    "vLLM": runner_dir / "run_vllm_offline.py",
}
# 每次重新生成实验矩阵都分配新 session；dry-run 不创建目录或结果文件。
run_id = uuid.uuid4().hex
result_path = runner_dir / "results" / run_id / "offline_results.jsonl"
engine_pythons, environment_issues = validate_engine_pythons()

commands = []
for engine, runner in runners.items():
    python_path = engine_pythons[engine]
    executable = str(python_path) if python_path is not None else f"<缺少 {engine_env_vars[engine]}>"
    for case in cases.to_dict(orient="records"):
        for run in range(1, 4):
            commands.append([
                executable, str(runner),
                "--model", str(model_dir),
                "--num-requests", str(case["num_requests"]),
                "--input-len", str(case["input_len"]),
                "--output-len", str(case["output_len"]),
                "--max-model-len", "4096",
                "--dtype", "bfloat16",
                "--results-path", str(result_path),
                "--notes", f"session={run_id} case={case['case']} run={run}",
            ])

print("session:", run_id)
print("results:", result_path)
print("commands:", len(commands))
print("first command:", shlex.join(commands[0]))
for issue in environment_issues:
    print("待配置:", issue)

## 6. 顺序执行

框架不能同时加载。子进程结束后，GPU 资源随进程释放。

In [ ]:
import subprocess

run_benchmarks = os.getenv("RUN_FRAMEWORK_BENCHMARKS") == "1"
current_pythons, environment_issues = validate_engine_pythons()
if run_benchmarks:
    if environment_issues:
        raise ValueError("拒绝执行：" + "; ".join(environment_issues))
    if current_pythons != engine_pythons:
        raise ValueError("Python 环境配置已变化，请重新生成命令和 session。")
    if not Path(model_dir).is_dir():
        raise FileNotFoundError("snapshot_download 返回的模型目录不存在。")
    for engine, runner in runners.items():
        if not runner.is_absolute() or not runner.is_file():
            raise FileNotFoundError(f"runner 不存在：{runner}")
    for command in commands:
        python_path = Path(command[0])
        if not python_path.is_absolute() or not python_path.is_file():
            raise ValueError(f"无效 Python 路径：{command[0]}")
        if python_path not in current_pythons.values():
            raise ValueError("命令使用了未配置的 Python。请重新生成命令。")
    executed_run_ids = globals().setdefault("executed_run_ids", set())
    if result_path.exists() or run_id in executed_run_ids:
        raise RuntimeError("本 session 已执行或已有结果，请重新生成命令取得新 run_id。")
    executed_run_ids.add(run_id)
    launch_failures = []
    for index, command in enumerate(commands, start=1):
        print(f"[{index}/{len(commands)}]", command[-1])
        try:
            completed = subprocess.run(command, cwd=repo_root, check=False)
            if completed.returncode != 0:
                launch_failures.append((command[-1], completed.returncode))
        except OSError as exc:
            launch_failures.append((command[-1], repr(exc)))
    print("子进程启动/退出失败（另计，不作为性能记录）:", launch_failures)
else:
    print("Dry-run：未启动任何框架子进程。")
    for issue in environment_issues:
        print("待配置:", issue)
    print("配置四个独立 Python 后，设置 RUN_FRAMEWORK_BENCHMARKS=1 并重新生成命令。")

## 7. 加载结果

缺少结果文件表示实验尚未运行，不会自动填入零值。

In [ ]:
import json
import re

def load_jsonl(path):
    if not path.exists():
        return []
    return [json.loads(line) for line in path.read_text().splitlines() if line.strip()]

records = load_jsonl(result_path)
results = pd.DataFrame(records)
print("records:", len(results))
results.head() if not results.empty else pd.DataFrame(columns=["engine", "case", "output_tokens_per_s"] )

## 8. 汇总三轮中位数

只汇总本轮 session 的成功记录。失败单独计数，不混入中位数或峰值；外部 session 和格式不符记录不参与本轮统计。旧字段 `peak_memory_gb` 保留兼容，展示名为 `caller_peak_allocated_gib`（GiB），只反映调用进程 allocator，不一定覆盖 worker，不是引擎总显存。

In [ ]:
import math
import re

notes_pattern = re.compile(r"session=([0-9a-f]{32}) case=([A-D]) run=([1-3])")
parsed_records = []
for record in records:
    row = dict(record)
    config = row.get("config")
    notes = config.get("notes", "") if isinstance(config, dict) else ""
    match = notes_pattern.fullmatch(notes) if isinstance(notes, str) else None
    row["session"], row["case"], row["run"] = (
        (match[1], match[2], int(match[3])) if match else (None, None, None)
    )
    valid_metrics = all(
        isinstance(row.get(field), (int, float))
        and not isinstance(row.get(field), bool)
        and math.isfinite(row[field]) and row[field] > 0
        for field in ("output_tokens_per_s", "elapsed_s")
    )
    row["failed"] = (
        row.get("error", "missing error field") is not None
        or bool(row.get("oom", False))
        or not valid_metrics
    )
    row.setdefault("peak_memory_gb", None)
    parsed_records.append(row)

result_columns = [
    "session", "case", "run", "engine", "failed", "error",
    "output_tokens_per_s", "elapsed_s", "peak_memory_gb",
]
results = pd.DataFrame(parsed_records) if parsed_records else pd.DataFrame(columns=result_columns)
session_results = results.loc[results["session"].eq(run_id)].copy()
successes = session_results.loc[session_results["failed"].eq(False)]
failures = session_results.loc[session_results["failed"].eq(True)]
print("本轮成功:", len(successes), "本轮失败:", len(failures))
print("外部 session 或格式不符（不参与汇总）:", len(results) - len(session_results))

summary_columns = [
    "case", "engine", "median_out_tok_s", "median_elapsed_s",
    "caller_peak_allocated_gib", "errors",
]
if session_results.empty:
    summary = pd.DataFrame(columns=summary_columns)
else:
    groups = session_results.groupby(["case", "engine"], dropna=False)
    error_counts = groups["failed"].sum().rename("errors")
    metrics = successes.groupby(["case", "engine"], dropna=False).agg(
        median_out_tok_s=("output_tokens_per_s", "median"),
        median_elapsed_s=("elapsed_s", "median"),
        caller_peak_allocated_gib=("peak_memory_gb", "max"),
    )
    summary = error_counts.to_frame().join(metrics).reset_index()[summary_columns]
summary

## 9. 结果完整性检查

完整实验应包含 4 个框架 × 4 个 case × 3 次运行，共 48 条记录。

条数不是充分条件：必须恰好覆盖全部唯一 `(engine, case, run)`，无重复、无外部 session、无 error/OOM/无效指标；启动或退出失败也需要单独排查。

In [ ]:
from collections import Counter
from itertools import product

expected_combinations = set(product(runners, ("A", "B", "C", "D"), (1, 2, 3)))
observed_counts = Counter(
    (row.get("engine"), row["case"], row["run"])
    for row in parsed_records
)
observed_combinations = set(observed_counts)
duplicates = {key: count for key, count in observed_counts.items() if count > 1}
missing = expected_combinations - observed_combinations
unexpected = observed_combinations - expected_combinations
all_current_session = all(row["session"] == run_id for row in parsed_records)
all_successful = all(not row["failed"] for row in parsed_records)
current_launch_failures = [
    failure for failure in globals().get("launch_failures", [])
    if failure[0].startswith(f"session={run_id} ")
]
complete = (
    observed_combinations == expected_combinations
    and not duplicates
    and all_current_session
    and all_successful
    and not current_launch_failures
)
print("expected unique combinations:", len(expected_combinations))
print("actual records:", len(parsed_records))
print("missing:", sorted(missing))
print("unexpected:", sorted(unexpected, key=str))
print("duplicates:", duplicates)
print("all current session:", all_current_session)
print("failed records:", sum(row["failed"] for row in parsed_records))
print("subprocess failures:", current_launch_failures)
print("complete:", complete)

## 10. 在线服务命令

服务横评单独覆盖 mini-SGLang 与 vLLM。两个服务不能同时占用同一张 GPU。

In [ ]:
serving_client = runner_dir / "run_serving_client.py"
serving_commands = {
    "mini-SGLang": [
        str(engine_pythons["mini-SGLang"]) if engine_pythons["mini-SGLang"] is not None else "<缺少 MINISGL_PYTHON>",
        str(serving_client),
        "--url", "http://127.0.0.1:1919/v1/chat/completions",
        "--endpoint", "chat", "--model", "Qwen/Qwen3-0.6B",
        "--num-requests", "16", "--concurrency", "4",
        "--prompt-words", "128", "--max-tokens", "128",
    ],
    "vLLM": [
        str(engine_pythons["vLLM"]) if engine_pythons["vLLM"] is not None else "<缺少 VLLM_PYTHON>",
        str(serving_client),
        "--url", "http://127.0.0.1:8000/v1/chat/completions",
        "--endpoint", "chat", "--model", "Qwen/Qwen3-0.6B",
        "--num-requests", "16", "--concurrency", "4",
        "--prompt-words", "128", "--max-tokens", "128",
    ],
}
for engine, command in serving_commands.items():
    print(engine, " ".join(command))

mini-SGLang 固定版本的流式接口不返回精确 usage token 数。客户端会记录 `output_unit_source=stream_chunks`，该 TPOT 不能与精确 token 口径混写。

## 11. 能力边界

性能数据与能力定位需要同时保留。

In [ ]:
capabilities = pd.DataFrame([
    ("Transformers", "模型库与正确性参照", "静态 generate", "无内置 serving API", "模型结构学习"),
    ("nano-vLLM", "推理引擎核心教材", "waiting/running scheduler", "未内置", "scheduler / KV block"),
    ("mini-SGLang", "轻量 serving 系统", "prefill/decode manager", "OpenAI-compatible", "Radix / overlap / serving"),
    ("vLLM", "生产级推理框架", "生产级 scheduler", "OpenAI-compatible", "部署与完整生态"),
], columns=["path", "position", "scheduling", "service", "best for"])
capabilities

## 12. 结论

```text
Transformers -> 模型加载、基础生成与正确性参照
nano-vLLM    -> 推理引擎核心机制学习
mini-SGLang  -> serving 系统结构学习
vLLM         -> 生产推理与服务能力
```

同机实测完成前，Notebook 不生成框架性能排名。完整结果需要 48 条离线记录，并额外保存在线服务的 TTFT、TPOT、吞吐、错误率和输出单位来源。